[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Excel Files


## What you will be able to do

Read and write `.xlsx` files, reach any sheet in a workbook by name rather than only the
first, and explain the two things that surprise everyone: a date that arrives as a
five-digit number, and a formula that reads as `None`.


## The idea

### The problem

CSV and JSON are formats a program produced. Excel is the format a **person** produced, and that
difference is the whole of this notebook.

A spreadsheet has things no plain format has: several sheets, formulas, merged cells, formatting
that carries meaning, and a header row that might start on line 4 because someone put a title
above it. It also has no schema, so a column of numbers can contain a stray note in row 200 and
nothing prevented that.

Two surprises account for most of the confusion. Dates sometimes arrive as numbers like `46082`.
And a cell containing a formula reads back as the formula text, or as `None`, but not as the
answer you can see on screen.

Neither is a bug. Both follow from how the format works.

### What an xlsx file is

> An **`.xlsx` file** is a zip archive of XML files. A **workbook** holds one or more
> **worksheets**, each a grid of **cells** addressed by column letter and row number.
>
> `openpyxl` reads and writes them. It is not part of the standard library and has to be
> installed, though Colab has it already.

Each cell stores a value **and** a number format. The value is what is on the disk; the format is
how Excel displays it. That separation is where the dates go wrong.

### Why a date arrives as a number

Excel does not store dates. It stores the number of days since 30 December 1899, and a **number
format** telling it to display that number as a date.

`46082` and `2026-03-01` are the same cell. Which one you get depends entirely on whether the
format survived, and formats are lost routinely: by an export, by a copy and paste, by a tool
that wrote the file without setting one.

`openpyxl` converts a cell to a `datetime` when the format says it is a date, and hands you the
raw number when it does not. The number is not corrupted data; it is the data, without the note
saying how to read it.

### Where you will meet this

Whenever a person sends you data. Survey exports, finance reports, lab records and anything
that has been through a spreadsheet on the way to you.

The **Pandas** guide reads Excel in one line and handles most of this. This notebook is what
that line is doing, which is what you need when it does the wrong thing.

### What this notebook covers

- Writing a workbook with more than one sheet, and reading it back
- Addressing cells by coordinate and by row and column number
- Iterating rows, with and without the cell objects
- The type each cell comes back as, and the number format beside it
- The date-as-a-number problem, produced and then converted back
- Formulas, and why `data_only=True` can give you `None`
- Finding a header that does not start in row 1
- Three errors, plus a type change that raises nothing

### A first look

Nothing to run yet.

```python
import openpyxl

wb = openpyxl.load_workbook("readings.xlsx")
ws = wb["Readings"]

print(ws["A2"].value)          # 2026-03-01 00:00:00, if the format survived
print(ws["A2"].number_format)  # 'yyyy-mm-dd'
```

Two cells can hold the identical date and come back as a `datetime` and as `46082`, differing
only in that second line.


## Setup

Four imports and a folder to work in.

- `openpyxl` reads and writes `.xlsx` files, and is the subject of this notebook
- `datetime` supplies the dates that demonstrate the number-format problem
- `Path` builds paths and checks the files exist
- `shutil` removes the scratch folder at the end

`openpyxl` is a third-party package. Colab has it already; on your own machine install it with
`pip install openpyxl`, which the **Environments and pip** notebook covered.

**Run this cell before the rest of the notebook.**


In [1]:

import openpyxl
import datetime
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print("openpyxl", openpyxl.__version__, "| working in:", scratch)


openpyxl 3.1.5 | working in: scratch


## Worked examples

### Writing a workbook

A new `Workbook` starts with one sheet. `append` adds a row to the end.


In [2]:

from openpyxl import Workbook

book = scratch / "readings.xlsx"

wb = Workbook()
sheet = wb.active
sheet.title = "Readings"

sheet.append(["date", "region", "value", "checked"])
sheet.append([datetime.date(2026, 3, 1), "north", 18.5, True])
sheet.append([datetime.date(2026, 3, 2), "south", 22.1, False])
sheet.append([datetime.date(2026, 3, 3), "north", 19.2, True])

notes = wb.create_sheet("Notes")
notes["A1"] = "Written by the Excel Files notebook."

wb.save(book)

print("sheets:", wb.sheetnames)
print("file size:", book.stat().st_size, "bytes")


sheets: ['Readings', 'Notes']
file size: 5491 bytes


Nothing is written to disk until `save`. A workbook you forget to save is lost exactly as
quietly as an unclosed file in the **Reading and Writing Text** notebook.

### Reading it back


In [3]:

wb = openpyxl.load_workbook(book)

print("sheets:", wb.sheetnames)

sheet = wb["Readings"]
print("dimensions:", sheet.dimensions)
print("rows:", sheet.max_row, "| columns:", sheet.max_column)


sheets: ['Readings', 'Notes']
dimensions: A1:D4
rows: 4 | columns: 4


`wb["Readings"]` gets a sheet by name, which survives someone reordering them. `wb.active` gets
whichever was selected when the file was saved, which is whatever the last person happened to be
looking at.

Two ways to iterate, and the difference matters.


In [4]:

for row in sheet.iter_rows(values_only=True):
    print(row)


('date', 'region', 'value', 'checked')
(datetime.datetime(2026, 3, 1, 0, 0), 'north', 18.5, True)
(datetime.datetime(2026, 3, 2, 0, 0), 'south', 22.1, False)
(datetime.datetime(2026, 3, 3, 0, 0), 'north', 19.2, True)


`values_only=True` gives plain tuples. Without it you get cell objects, which carry the format,
the coordinate and everything else.


In [5]:

for row in sheet.iter_rows(min_row=1, max_row=2):
    for cell in row:
        print(f"{cell.coordinate:<4} {cell.value!r:<28} {type(cell.value).__name__}")
    print()


A1   'date'                       str
B1   'region'                     str
C1   'value'                      str
D1   'checked'                    str

A2   datetime.datetime(2026, 3, 1, 0, 0) datetime
B2   'north'                      str
C2   18.5                         float
D2   True                         bool



### Addressing cells

Two ways, and both are useful.


In [6]:

print("by coordinate:", sheet["B2"].value)
print("by numbers:   ", sheet.cell(row=2, column=2).value)


by coordinate: north
by numbers:    north


Coordinates read like a spreadsheet and are what you use when a person told you where something
is. Row and column numbers are what you use in a loop.

Both are **one-based**, unlike everything else in Python. `sheet.cell(row=1, column=1)` is `A1`,
and there is no row 0. That is a real inconsistency and it catches people who assume the
zero-based indexing from the **Lists** notebook applies here.


### The date that arrives as a number

Here is the promise from the **JSON on Disk** notebook, produced deliberately.


In [7]:

dates = scratch / "dates.xlsx"

wb = Workbook()
sheet = wb.active

sheet["A1"] = datetime.date(2026, 3, 1)
sheet["A2"] = datetime.date(2026, 3, 1)
sheet["A2"].number_format = "General"      # the format someone lost

wb.save(dates)

sheet = openpyxl.load_workbook(dates).active

for coord in ["A1", "A2"]:
    cell = sheet[coord]
    print(f"{coord}: {cell.value!r:<34} {type(cell.value).__name__:<9} format={cell.number_format!r}")


A1: datetime.datetime(2026, 3, 1, 0, 0) datetime  format='yyyy-mm-dd'
A2: 46082                              int       format='General'


The same date, written twice, comes back as a `datetime` and as the integer `46082`. The only
difference is the number format, and clearing it is something an export or a copy and paste does
without asking.

`46082` is the number of days since 30 December 1899. Converting it back is arithmetic:


In [8]:

from openpyxl.utils.datetime import from_excel

serial = sheet["A2"].value

print("openpyxl's converter:", from_excel(serial, openpyxl.load_workbook(dates).epoch))

epoch = datetime.datetime(1899, 12, 30)
print("by hand:             ", (epoch + datetime.timedelta(days=serial)).date())


openpyxl's converter: 2026-03-01 00:00:00
by hand:              2026-03-01


Both give 1 March 2026.

The epoch is 30 December rather than 31 December for a reason worth knowing. Excel believes 1900
was a leap year, which it was not, so its calendar contains a 29 February 1900 that never
happened. The odd epoch cancels that out for every date after February 1900.

It does not cancel out before then, and the two methods disagree there:


In [9]:

for serial in [1, 59, 60, 61]:
    converted = from_excel(serial, openpyxl.load_workbook(dates).epoch)
    by_hand = (epoch + datetime.timedelta(days=serial)).date()
    print(f"serial {serial:<3} openpyxl {converted.date()}   by hand {by_hand}")


serial 1   openpyxl 1900-01-01   by hand 1899-12-31
serial 59  openpyxl 1900-02-28   by hand 1900-02-27
serial 60  openpyxl 1900-02-28   by hand 1900-02-28
serial 61  openpyxl 1900-03-01   by hand 1900-03-01


For serials 1 and 59 the hand calculation is a day early. Serials 59 and 60 both come back as
28 February 1900, because 60 is Excel's day that does not exist.

None of this matters for data from this century. Use `from_excel` and it is handled; do the
arithmetic yourself and know it is only reliable from March 1900 onward.


### Formulas

A cell holding a formula does not hold its result.


In [10]:

formulas = scratch / "formulas.xlsx"

wb = Workbook()
sheet = wb.active
sheet["A1"] = 10
sheet["A2"] = 20
sheet["A3"] = "=SUM(A1:A2)"
wb.save(formulas)

sheet = openpyxl.load_workbook(formulas).active
print("A3 value:", repr(sheet["A3"].value))


A3 value: '=SUM(A1:A2)'


The formula text, not `30`. `openpyxl` does not calculate anything; it reads and writes the
file, and calculation is Excel's job.

`data_only=True` asks for the **cached** result Excel stored the last time it opened the file.


In [11]:

cached = openpyxl.load_workbook(formulas, data_only=True).active

print("with data_only=True:", repr(cached["A3"].value))


with data_only=True: None


`None`. This file was written by `openpyxl` and never opened by Excel, so there is no cached
value to read.

That is the trap. `data_only=True` gives the right answer for a file a person saved from Excel,
and `None` for one a program produced. A script that works on files from colleagues and returns
`None` on files it generated itself is showing you this, not a bug in your code.

If you need the number, either compute it in Python or open and resave the file in Excel once.


### A header that does not start in row 1

Spreadsheets made by people often have a title, a blank line, then the real header.


In [12]:

messy = scratch / "messy.xlsx"

wb = Workbook()
sheet = wb.active
sheet["A1"] = "Regional readings, March 2026"
sheet["A3"] = "region"
sheet["B3"] = "value"
sheet["A4"] = "north"
sheet["B4"] = 18.5
sheet["A5"] = "south"
sheet["B5"] = 22.1
wb.save(messy)

sheet = openpyxl.load_workbook(messy).active

for row in sheet.iter_rows(values_only=True):
    print(row)


('Regional readings, March 2026', None)
(None, None)
('region', 'value')
('north', 18.5)
('south', 22.1)


Reading that with the first row as the header gives a column called
`Regional readings, March 2026` and one called `None`.

Find the header rather than assuming it:


In [13]:

expected = {"region", "value"}
header_row = None

for row in sheet.iter_rows(values_only=True):
    if expected <= {str(v).strip() for v in row if v is not None}:
        header_row = row
        break

print("header found:", header_row)


header found: ('region', 'value')


`expected <= {...}` is the subset test from the **Sets** notebook: is every column I need present
in this row. It finds the header wherever it is and fails loudly if the file does not contain
one, which is better than reading the wrong row silently.


### Cells that are empty, and cells that are not there

An empty cell reads as `None`. So does a cell outside the used range.


In [14]:

sheet = openpyxl.load_workbook(book)["Readings"]

print("A2 (has data):", repr(sheet["A2"].value))
print("Z99 (nothing):", repr(sheet["Z99"].value))
print("max_row is still:", sheet.max_row)


A2 (has data): datetime.datetime(2026, 3, 1, 0, 0)
Z99 (nothing): None
max_row is still: 99


Asking for `Z99` did not raise and did not extend the sheet. There is no `IndexError` for a cell
that does not exist, which means a typo in a coordinate produces `None` rather than an error.

That is worth remembering: `sheet["B2"].value` and `sheet["BB2"].value` both look plausible, and
only one of them is your data.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/06-excel-files-solutions.ipynb).

**1.** Create a workbook with a sheet named `Cities`, write a header and three rows, save it to
`scratch/cities.xlsx`, and print the sheet names.


In [15]:
# your code here


**2.** Load it back and print every row using `iter_rows(values_only=True)`.


In [16]:
# your code here


**3.** Print the value and the type of cell `B2`, addressed both by coordinate and by row and
column number.


In [17]:
# your code here


**4.** Add a second sheet called `Summary`, write the number of cities into `A1`, and save.
Print the sheet names to confirm.


In [18]:
# your code here


**5.** Write today's date into a cell, set its `number_format` to `"General"`, save, reload, and
print what comes back.


In [19]:
# your code here


**6.** Convert the number from task 5 back into a date.


In [20]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### FileNotFoundError: load_workbook does not create files


In [21]:

openpyxl.load_workbook(scratch / "not_here.xlsx")


FileNotFoundError: [Errno 2] No such file or directory: 'scratch/not_here.xlsx'

Unlike `open` with mode `"w"`, there is no version of `load_workbook` that makes a file. Use
`Workbook()` to create one, and `load_workbook` only to read an existing one.


### KeyError: the sheet is not called that


In [22]:

wb = openpyxl.load_workbook(book)

wb["readings"]


KeyError: 'Worksheet readings does not exist.'

`Worksheet readings does not exist`. Sheet names are case sensitive, and the file has
`Readings`.

They also often carry trailing spaces, because a person typed them. `wb.sheetnames` with `repr`
shows exactly what they are:


In [23]:

print([repr(name) for name in wb.sheetnames])


["'Readings'", "'Notes'"]


### TypeError: openpyxl cannot store that

Cells hold numbers, strings, booleans, dates and `None`. Nothing else.


In [24]:

wb = Workbook()
wb.active["A1"] = [1, 2, 3]


ValueError: Cannot convert [1, 2, 3] to Excel

`Cannot convert [1, 2, 3] to Excel`. A spreadsheet cell has no concept of a list, so there is no
sensible conversion and it raises rather than choosing one, which is the same behavior JSON had
for a set.

Decide how it should look and write that:


In [25]:

wb.active["A1"] = ", ".join(str(n) for n in [1, 2, 3])

print(repr(wb.active["A1"].value))


'1, 2, 3'


### The quiet one: a date became a number and the arithmetic carried on


In [26]:

report = scratch / "report.xlsx"

wb = Workbook()
sheet = wb.active
sheet["A1"] = "recorded"
sheet["A2"] = datetime.date(2026, 3, 1)
sheet["A2"].number_format = "General"
sheet["B2"] = 5
wb.save(report)

sheet = openpyxl.load_workbook(report).active
recorded = sheet["A2"].value
days = sheet["B2"].value

print("recorded:", repr(recorded))
print("recorded + days:", recorded + days)


recorded: 46082
recorded + days: 46087


`46087`, and no error anywhere. The addition was legal because both values were numbers, and the
result is meaningless.

Had the format survived, `recorded` would have been a `datetime` and `datetime + int` would have
raised immediately.

The defense is to check the type on the way in rather than trusting the file:


In [27]:

if isinstance(recorded, (datetime.date, datetime.datetime)):
    print("got a date:", recorded)
else:
    print("got a", type(recorded).__name__, "- converting")
    recorded = (datetime.datetime(1899, 12, 30) + datetime.timedelta(days=recorded)).date()
    print("converted:", recorded)


got a int - converting
converted: 2026-03-01


### Cleaning up


In [28]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- An `.xlsx` file is a workbook of worksheets of cells; `openpyxl` reads and writes them.
- `Workbook()` creates, `load_workbook` reads, and nothing reaches the disk until `save`.
- Get sheets by name rather than `wb.active`, which depends on what was selected when saved.
- Cell addressing is **one-based**: `cell(row=1, column=1)` is `A1`.
- Excel stores a date as days since 30 December 1899 plus a number format. Lose the format and
  you get the number.
- `from_excel` converts it back; hand arithmetic is only reliable after February 1900.
- A formula cell holds the formula. `data_only=True` gives the cached result, which is `None`
  for a file Excel has never opened.
- Any cell outside the data reads as `None` rather than raising, so a mistyped coordinate is
  silent.
- Check the type of what you read. A date that arrived as a number will do arithmetic happily.


## What is next

The **Directories** notebook, which stops working on one file at a time. Walking a folder tree,
matching names by pattern, and building a record of what you found before you process any of it.


---

&#8592; **Previous:** [JSON on Disk](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/05-json-on-disk.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
